# LiteRCF-Net: Lightweight Region–Context–Frequency Thyroid Classification

**Purpose:** a Colab-ready research pipeline for a single free NVIDIA T4 GPU.

### Main experiment
- Train and test on the **official TN5000 split** whenever the split files are available.
- Use the second folder-based thyroid dataset only for **external evaluation**.
- Preserve checkpoints, tables, plots, and predictions in Google Drive.

### Proposed model
`Shared MobileNetV4-Conv-Small → lesion ROI + peri-lesional context → fixed Haar frequency branch → gated fusion → calibrated classifier`

### Before running
In Colab, select **Runtime → Change runtime type → T4 GPU**, then run every cell in order.

The default experiment is the full proposed model. For ablation studies, change `EXPERIMENT_NAME` in Step 2 and rerun the notebook with a new output folder.

## Step 1 — Install the small set of additional packages

Colab already provides PyTorch, TorchVision, OpenCV, NumPy, and scikit-learn. This cell installs:
- `timm` for pretrained MobileNetV4
- `kaggle` for dataset download
- `thop` for approximate FLOP counting

**Desired output:** installation finishes without an error.

In [ ]:
!pip -q install -U timm kaggle thop

## Step 2 — Imports, Google Drive, configuration, and GPU audit

The default batch size is deliberately conservative because the model processes:
1. lesion ROI,
2. context crop,
3. clean view,
4. marker-perturbed view.

With mixed precision, `BATCH_SIZE=8` and `ACCUM_STEPS=2` are suitable starting values for a free T4.

**Desired output:** one CUDA GPU is detected and the device name contains `T4`.

In [ ]:
import os
import gc
import cv2
import json
import math
import time
import copy
import random
import hashlib
import warnings
import xml.etree.ElementTree as ET
from pathlib import Path
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, matthews_corrcoef,
    confusion_matrix, roc_curve, precision_recall_curve,
    brier_score_loss
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms.functional as TVF
from torchvision.transforms import InterpolationMode

import timm
from thop import profile
from google.colab import drive, files

warnings.filterwarnings("ignore")
drive.mount("/content/drive")

# --------------------------- USER CONFIGURATION ---------------------------
SEED = 42
IMAGE_SIZE = 224
BATCH_SIZE = 8
ACCUM_STEPS = 2
NUM_WORKERS = 2

WARMUP_EPOCHS = 5
FINETUNE_EPOCHS = 25
EARLY_STOPPING_PATIENCE = 7

CONTEXT_EXPANSION = 0.25
USE_AMP = True
RUN_EXACT_DUPLICATE_AUDIT = True

# Choose one:
# A0_roi_only, A1_roi_context, A2_roi_frequency, A3_no_consistency, full
EXPERIMENT_NAME = "full"

ABLATIONS = {
    "A0_roi_only": {
        "use_context": False,
        "use_frequency": False,
        "lambda_consistency": 0.0
    },
    "A1_roi_context": {
        "use_context": True,
        "use_frequency": False,
        "lambda_consistency": 0.0
    },
    "A2_roi_frequency": {
        "use_context": False,
        "use_frequency": True,
        "lambda_consistency": 0.0
    },
    "A3_no_consistency": {
        "use_context": True,
        "use_frequency": True,
        "lambda_consistency": 0.0
    },
    "full": {
        "use_context": True,
        "use_frequency": True,
        "lambda_consistency": 0.15
    }
}
EXP = ABLATIONS[EXPERIMENT_NAME]

DRIVE_ROOT = Path("/content/drive/MyDrive/LiteRCF_Thyroid")
RUN_DIR = DRIVE_ROOT / f"{EXPERIMENT_NAME}_seed{SEED}"
DATA_DIR = Path("/content/thyroid_data")
TN_DOWNLOAD_DIR = DATA_DIR / "tn5000"
EXT_DOWNLOAD_DIR = DATA_DIR / "external"

for directory in [RUN_DIR, DATA_DIR, TN_DOWNLOAD_DIR, EXT_DOWNLOAD_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert DEVICE.type == "cuda", "Enable a GPU runtime before continuing."

print("=" * 72)
print("RUNTIME AND EXPERIMENT CONFIGURATION")
print("=" * 72)
print("PyTorch version       :", torch.__version__)
print("CUDA available        :", torch.cuda.is_available())
print("GPU                    :", torch.cuda.get_device_name(0))
print("GPU memory             :", f"{torch.cuda.get_device_properties(0).total_memory / 2**30:.2f} GB")
print("Experiment             :", EXPERIMENT_NAME)
print("Seed                   :", SEED)
print("Batch / accumulation   :", BATCH_SIZE, "/", ACCUM_STEPS)
print("Effective batch size   :", BATCH_SIZE * ACCUM_STEPS)
print("Output folder          :", RUN_DIR)
print("Experiment components  :", EXP)

## Step 3 — Reproducibility helpers

The free T4 is used for speed, while seeds are fixed. For the final paper, rerun the full model with at least seeds `42`, `43`, and `44`.

**Desired output:** the selected seed is printed.

In [ ]:
def seed_everything(seed):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.benchmark = True
    torch.backends.cudnn.deterministic = False

seed_everything(SEED)
print(f"Random seed fixed to {SEED}.")

## Step 4 — Kaggle credentials and dataset download

This downloads the TN5000-style primary dataset and the folder-based external dataset. Configure `KAGGLE_USERNAME` and `KAGGLE_KEY` as private runtime environment variables before running; credentials are never stored in this notebook.


In [ ]:
if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
    raise RuntimeError("Set KAGGLE_USERNAME and KAGGLE_KEY in the runtime environment.")

def directory_has_files(path):
    return any(p.is_file() for p in Path(path).rglob("*"))

if not directory_has_files(TN_DOWNLOAD_DIR):
    subprocess.run(["kaggle", "datasets", "download", "-d", "abdullahelafifi/main-data", "-p", str(TN_DOWNLOAD_DIR), "--unzip"], check=True)
if not directory_has_files(EXT_DOWNLOAD_DIR):
    subprocess.run(["kaggle", "datasets", "download", "-d", "sowmyaabirami/thyroid-ultrasound-dataset", "-p", str(EXT_DOWNLOAD_DIR), "--unzip"], check=True)
print("TN5000 files :", sum(p.is_file() for p in TN_DOWNLOAD_DIR.rglob("*")))
print("External files:", sum(p.is_file() for p in EXT_DOWNLOAD_DIR.rglob("*")))


## Step 5 — Locate TN5000 folders and parse XML annotations

The code automatically finds `JPEGImages`, `Annotations`, and `ImageSets/Main`.  
Each XML file supplies:
- binary label,
- lesion bounding box,
- image identifier.

If official `train.txt`, `val.txt`, and `test.txt` files are available, they are used. Otherwise, the cell creates and saves a deterministic 70/10/20 stratified fallback split and prints a warning.

**Desired output:** approximately 5,000 usable TN5000 rows and split counts close to 3,500/500/1,000.

In [ ]:
IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}

def find_tn_root(search_root):
    search_root = Path(search_root)
    for image_dir in search_root.rglob("JPEGImages"):
        candidate = image_dir.parent
        if (candidate / "Annotations").exists():
            return candidate
    raise FileNotFoundError("Could not find a folder containing JPEGImages and Annotations.")

TN_ROOT = find_tn_root(TN_DOWNLOAD_DIR)
TN_IMAGE_DIR = TN_ROOT / "JPEGImages"
TN_ANNOT_DIR = TN_ROOT / "Annotations"
TN_SET_DIR = TN_ROOT / "ImageSets" / "Main"

image_lookup = {
    p.stem: p for p in TN_IMAGE_DIR.iterdir()
    if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
}

def parse_xml(xml_path):
    root = ET.parse(xml_path).getroot()
    objects = []
    for obj in root.findall("object"):
        name_node = obj.find("name")
        box_node = obj.find("bndbox")
        if name_node is None or box_node is None:
            continue

        raw_name = str(name_node.text).strip().lower()
        label_map = {
            "0": 0, "benign": 0, "b": 0,
            "1": 1, "malignant": 1, "m": 1
        }
        if raw_name not in label_map:
            continue

        try:
            box = {
                "xmin": float(box_node.find("xmin").text),
                "ymin": float(box_node.find("ymin").text),
                "xmax": float(box_node.find("xmax").text),
                "ymax": float(box_node.find("ymax").text)
            }
        except Exception:
            continue

        objects.append((label_map[raw_name], box))

    if not objects:
        return None

    labels = [item[0] for item in objects]
    label = max(set(labels), key=labels.count)
    selected_boxes = [box for obj_label, box in objects if obj_label == label]

    union_box = {
        "xmin": min(box["xmin"] for box in selected_boxes),
        "ymin": min(box["ymin"] for box in selected_boxes),
        "xmax": max(box["xmax"] for box in selected_boxes),
        "ymax": max(box["ymax"] for box in selected_boxes)
    }
    return label, union_box

records = []
for image_id, image_path in image_lookup.items():
    xml_path = TN_ANNOT_DIR / f"{image_id}.xml"
    if not xml_path.exists():
        continue
    parsed = parse_xml(xml_path)
    if parsed is None:
        continue
    label, box = parsed
    records.append({
        "image_id": image_id,
        "path": str(image_path),
        "label": int(label),
        **box,
        "source": "TN5000"
    })

df_tn = pd.DataFrame(records).sort_values("image_id").reset_index(drop=True)
assert len(df_tn) > 0, "No TN5000 records were parsed."

def read_split_file(split_name):
    if not TN_SET_DIR.exists():
        return None

    preferred = TN_SET_DIR / f"{split_name}.txt"
    candidates = [preferred] if preferred.exists() else list(TN_SET_DIR.glob(f"*{split_name}*.txt"))

    for candidate in candidates:
        ids = []
        for line in candidate.read_text(errors="ignore").splitlines():
            token = line.strip().split()[0] if line.strip() else ""
            if token:
                ids.append(Path(token).stem)
        ids = [item for item in ids if item in image_lookup]
        if ids:
            print(f"{split_name}: using {candidate.name}")
            return set(ids)
    return None

train_ids = read_split_file("train")
val_ids = read_split_file("val")
test_ids = read_split_file("test")

official_available = all(x is not None for x in [train_ids, val_ids, test_ids])

if official_available:
    overlap = (train_ids & val_ids) | (train_ids & test_ids) | (val_ids & test_ids)
    assert not overlap, f"Official split files overlap: {len(overlap)} identifiers."
    split_map = {image_id: "train" for image_id in train_ids}
    split_map.update({image_id: "val" for image_id in val_ids})
    split_map.update({image_id: "test" for image_id in test_ids})
    df_tn["split"] = df_tn["image_id"].map(split_map)
    missing_split = df_tn["split"].isna().sum()
    if missing_split:
        print(f"Warning: {missing_split} parsed images were not listed in official split files.")
        df_tn = df_tn.dropna(subset=["split"]).reset_index(drop=True)
else:
    print("WARNING: complete official split files were not found.")
    print("A deterministic stratified 70/10/20 fallback split will be used.")
    train_df, temp_df = train_test_split(
        df_tn, test_size=0.30, random_state=SEED, stratify=df_tn["label"]
    )
    val_df, test_df = train_test_split(
        temp_df, test_size=2/3, random_state=SEED, stratify=temp_df["label"]
    )
    df_tn["split"] = ""
    df_tn.loc[train_df.index, "split"] = "train"
    df_tn.loc[val_df.index, "split"] = "val"
    df_tn.loc[test_df.index, "split"] = "test"

df_tn.to_csv(RUN_DIR / "tn5000_index_and_split.csv", index=False)

print("\nTN5000 root:", TN_ROOT)
print("Official split used:", official_available)
print("\nClass counts:")
print(df_tn["label"].value_counts().sort_index().rename(index={0: "Benign", 1: "Malignant"}))
print("\nSplit counts:")
print(pd.crosstab(df_tn["split"], df_tn["label"], margins=True))

## Step 6 — Index the external folder-based dataset

The external dataset is **not mixed into the primary TN5000 benchmark training split**. This avoids reporting an unfair result against papers that used only TN5000.

Because this dataset has no XML boxes, its images are treated as already nodule-focused:
- ROI = full image
- Context = full image

**Desired output:** benign and malignant image counts are printed.

In [ ]:
def find_class_folder(search_root, class_name):
    matches = [
        p for p in Path(search_root).rglob("*")
        if p.is_dir() and p.name.lower() == class_name.lower()
    ]
    if not matches:
        raise FileNotFoundError(f"Could not find a '{class_name}' folder.")
    return max(matches, key=lambda p: sum(1 for x in p.iterdir() if x.is_file()))

BENIGN_DIR = find_class_folder(EXT_DOWNLOAD_DIR, "benign")
MALIGNANT_DIR = find_class_folder(EXT_DOWNLOAD_DIR, "malignant")

external_records = []
for label, folder in [(0, BENIGN_DIR), (1, MALIGNANT_DIR)]:
    for image_path in folder.rglob("*"):
        if image_path.is_file() and image_path.suffix.lower() in IMAGE_EXTENSIONS:
            external_records.append({
                "image_id": f"EXT_{image_path.stem}",
                "path": str(image_path),
                "label": label,
                "xmin": np.nan,
                "ymin": np.nan,
                "xmax": np.nan,
                "ymax": np.nan,
                "source": "External",
                "split": "external"
            })

df_external = pd.DataFrame(external_records).drop_duplicates("path").reset_index(drop=True)
assert len(df_external) > 0, "No external images were found."

print("Benign folder   :", BENIGN_DIR)
print("Malignant folder:", MALIGNANT_DIR)
print("\nExternal class counts:")
print(df_external["label"].value_counts().sort_index().rename(index={0: "Benign", 1: "Malignant"}))

## Step 7 — Exact duplicate audit across datasets

This prevents identical external images from being counted as independent generalization evidence. Hashes are cached in Google Drive.

**Desired output:** the number of exact cross-dataset duplicates is printed. External duplicates, when found, are removed from external evaluation.

In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

hash_cache_path = RUN_DIR / "dataset_hashes.csv"

if RUN_EXACT_DUPLICATE_AUDIT:
    if hash_cache_path.exists():
        hash_df = pd.read_csv(hash_cache_path)
    else:
        hash_rows = []
        combined_for_hash = pd.concat([
            df_tn[["path", "source"]],
            df_external[["path", "source"]]
        ], ignore_index=True)

        for index, row in combined_for_hash.iterrows():
            if index % 1000 == 0:
                print(f"Hashing {index}/{len(combined_for_hash)}")
            hash_rows.append({
                "path": row["path"],
                "source": row["source"],
                "sha256": sha256_file(row["path"])
            })

        hash_df = pd.DataFrame(hash_rows)
        hash_df.to_csv(hash_cache_path, index=False)

    tn_hashes = set(hash_df.loc[hash_df["source"] == "TN5000", "sha256"])
    ext_hash_map = hash_df.loc[hash_df["source"] == "External", ["path", "sha256"]]
    duplicate_external_paths = set(
        ext_hash_map.loc[ext_hash_map["sha256"].isin(tn_hashes), "path"]
    )

    print("Exact cross-dataset duplicates:", len(duplicate_external_paths))
    if duplicate_external_paths:
        df_external = df_external[~df_external["path"].isin(duplicate_external_paths)].reset_index(drop=True)
        print("External rows after removal:", len(df_external))
else:
    print("Exact duplicate audit skipped by configuration.")

df_external.to_csv(RUN_DIR / "external_index_clean.csv", index=False)

## Step 8 — Image preprocessing, ROI/context extraction, marker perturbation, and Dataset class

Pipeline:
1. load ultrasound image,
2. crop annotated lesion ROI,
3. create 25% expanded peri-lesional context crop,
4. grayscale + median filter + CLAHE,
5. create a marker-perturbed consistency view,
6. apply shared geometric/photometric augmentation,
7. apply ImageNet normalization.

Vertical flipping is deliberately excluded because it can invert ultrasound depth orientation.

**Desired output:** the dataset class is defined without an error.

In [ ]:
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

def valid_bbox(row):
    values = [row["xmin"], row["ymin"], row["xmax"], row["ymax"]]
    return not any(pd.isna(value) for value in values)

def clip_box(box, width, height):
    xmin, ymin, xmax, ymax = box
    xmin = int(np.clip(round(xmin), 0, width - 1))
    ymin = int(np.clip(round(ymin), 0, height - 1))
    xmax = int(np.clip(round(xmax), xmin + 1, width))
    ymax = int(np.clip(round(ymax), ymin + 1, height))
    return xmin, ymin, xmax, ymax

def expand_box(box, width, height, expansion):
    xmin, ymin, xmax, ymax = box
    box_w = xmax - xmin
    box_h = ymax - ymin
    pad_x = box_w * expansion
    pad_y = box_h * expansion
    return clip_box(
        (xmin - pad_x, ymin - pad_y, xmax + pad_x, ymax + pad_y),
        width, height
    )

def crop_from_box(image, box):
    xmin, ymin, xmax, ymax = box
    crop = image[ymin:ymax, xmin:xmax]
    return crop if crop.size else image

def ultrasound_preprocess(image):
    image = cv2.resize(image, (IMAGE_SIZE, IMAGE_SIZE), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(image, cv2.COLOR_RGB2GRAY)
    gray = cv2.medianBlur(gray, 3)
    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    return cv2.cvtColor(enhanced, cv2.COLOR_GRAY2RGB)

def marker_variant(image):
    output = image.copy()
    gray = cv2.cvtColor(output, cv2.COLOR_RGB2GRAY)

    # Suppress long, thin, very bright horizontal/vertical structures.
    bright = (gray > 245).astype(np.uint8) * 255
    horizontal = cv2.morphologyEx(
        bright, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (17, 1))
    )
    vertical = cv2.morphologyEx(
        bright, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_RECT, (1, 17))
    )
    line_mask = cv2.dilate(
        cv2.bitwise_or(horizontal, vertical),
        np.ones((3, 3), np.uint8),
        iterations=1
    )

    if line_mask.any():
        output = cv2.inpaint(output, line_mask, 3, cv2.INPAINT_TELEA)

    # Add synthetic lines with equal probability across both classes.
    if random.random() < 0.50:
        h, w = output.shape[:2]
        x = random.randint(int(0.20 * w), int(0.80 * w))
        y = random.randint(int(0.20 * h), int(0.80 * h))
        length = random.randint(int(0.08 * w), int(0.20 * w))
        value = random.randint(210, 255)
        thickness = random.choice([1, 1, 2])
        cv2.line(output, (max(0, x - length), y), (min(w - 1, x + length), y),
                 (value, value, value), thickness)
        cv2.line(output, (x, max(0, y - length)), (x, min(h - 1, y + length)),
                 (value, value, value), thickness)

    return output

def shared_augment(images):
    h, w = images[0].shape[:2]

    if random.random() < 0.50:
        images = [cv2.flip(image, 1) for image in images]

    angle = random.uniform(-8.0, 8.0)
    scale = random.uniform(0.95, 1.05)
    tx = random.uniform(-0.04, 0.04) * w
    ty = random.uniform(-0.04, 0.04) * h

    matrix = cv2.getRotationMatrix2D((w / 2, h / 2), angle, scale)
    matrix[:, 2] += [tx, ty]
    images = [
        cv2.warpAffine(
            image, matrix, (w, h), flags=cv2.INTER_LINEAR,
            borderMode=cv2.BORDER_REFLECT_101
        )
        for image in images
    ]

    alpha = random.uniform(0.90, 1.10)
    beta = random.uniform(-12.0, 12.0)
    images = [
        np.clip(image.astype(np.float32) * alpha + beta, 0, 255).astype(np.uint8)
        for image in images
    ]
    return images

def image_to_tensor(image):
    array = image.astype(np.float32) / 255.0
    array = (array - IMAGENET_MEAN) / IMAGENET_STD
    return torch.from_numpy(array.transpose(2, 0, 1)).float()

class ThyroidRegionDataset(Dataset):
    def __init__(self, dataframe, train=False):
        self.df = dataframe.reset_index(drop=True).copy()
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):
        row = self.df.iloc[index]
        image_bgr = cv2.imread(row["path"])
        if image_bgr is None:
            raise FileNotFoundError(row["path"])
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        height, width = image.shape[:2]

        if valid_bbox(row):
            lesion_box = clip_box(
                (row["xmin"], row["ymin"], row["xmax"], row["ymax"]),
                width, height
            )
            context_box = expand_box(
                lesion_box, width, height, CONTEXT_EXPANSION
            )
            roi = crop_from_box(image, lesion_box)
            context = crop_from_box(image, context_box)
        else:
            roi = image
            context = image

        roi = ultrasound_preprocess(roi)
        context = ultrasound_preprocess(context)

        roi_marker = marker_variant(roi)
        context_marker = marker_variant(context)

        if self.train:
            roi, context, roi_marker, context_marker = shared_augment(
                [roi, context, roi_marker, context_marker]
            )

        return {
            "roi": image_to_tensor(roi),
            "context": image_to_tensor(context),
            "roi_marker": image_to_tensor(roi_marker),
            "context_marker": image_to_tensor(context_marker),
            "label": torch.tensor(int(row["label"]), dtype=torch.long),
            "path": row["path"]
        }

print("ThyroidRegionDataset is ready.")

## Step 9 — Create train, validation, test, and external DataLoaders

Only the training loader is shuffled and augmented. Validation/test/external loaders are deterministic.

**Desired output:** loader sizes and balanced class weights are printed.

In [ ]:
train_df = df_tn[df_tn["split"] == "train"].reset_index(drop=True)
val_df = df_tn[df_tn["split"] == "val"].reset_index(drop=True)
test_df = df_tn[df_tn["split"] == "test"].reset_index(drop=True)

train_dataset = ThyroidRegionDataset(train_df, train=True)
val_dataset = ThyroidRegionDataset(val_df, train=False)
test_dataset = ThyroidRegionDataset(test_df, train=False)
external_dataset = ThyroidRegionDataset(df_external, train=False)

loader_kwargs = {
    "batch_size": BATCH_SIZE,
    "num_workers": NUM_WORKERS,
    "pin_memory": True,
    "persistent_workers": NUM_WORKERS > 0
}

train_loader = DataLoader(train_dataset, shuffle=True, drop_last=True, **loader_kwargs)
val_loader = DataLoader(val_dataset, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_dataset, shuffle=False, drop_last=False, **loader_kwargs)
external_loader = DataLoader(external_dataset, shuffle=False, drop_last=False, **loader_kwargs)

weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0, 1]),
    y=train_df["label"].to_numpy()
)
CLASS_WEIGHTS = torch.tensor(weights, dtype=torch.float32, device=DEVICE)

print("Train / val / test / external:",
      len(train_dataset), len(val_dataset), len(test_dataset), len(external_dataset))
print("Train batches:", len(train_loader))
print("Class weights:", CLASS_WEIGHTS.detach().cpu().numpy())

## Step 10 — Visually verify ROI, context, and marker perturbation

Do not start training until these crops look correct.

**Desired output:** a figure with lesion ROI, expanded context, marker-perturbed ROI, and marker-perturbed context.

In [ ]:
sample = train_dataset[random.randrange(len(train_dataset))]

def denormalize(tensor):
    array = tensor.permute(1, 2, 0).numpy()
    array = array * IMAGENET_STD + IMAGENET_MEAN
    return np.clip(array, 0, 1)

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
titles = ["Lesion ROI", "Expanded context", "Marker view: ROI", "Marker view: context"]
keys = ["roi", "context", "roi_marker", "context_marker"]

for axis, title, key in zip(axes, titles, keys):
    axis.imshow(denormalize(sample[key]))
    axis.set_title(title)
    axis.axis("off")

plt.tight_layout()
plt.show()
print("Label:", "Malignant" if sample["label"].item() == 1 else "Benign")

## Step 11 — Define LiteRCF-Net

The same MobileNetV4 encoder processes both lesion and context crops. This reuses weights rather than maintaining two heavy backbones.

The frequency branch uses fixed Haar filters, so the transform itself adds no trainable parameters.

**Desired output:** model name, feature dimension, and parameter counts are printed.

In [ ]:
BACKBONE_ID = "hf_hub:timm/mobilenetv4_conv_small.e2400_r224_in1k"

class HaarFrequencyBranch(nn.Module):
    def __init__(self, output_dim=256):
        super().__init__()
        ll = torch.tensor([[1, 1], [1, 1]], dtype=torch.float32) / 2
        lh = torch.tensor([[-1, -1], [1, 1]], dtype=torch.float32) / 2
        hl = torch.tensor([[-1, 1], [-1, 1]], dtype=torch.float32) / 2
        hh = torch.tensor([[1, -1], [-1, 1]], dtype=torch.float32) / 2
        filters = torch.stack([ll, lh, hl, hh]).unsqueeze(1)
        self.register_buffer("haar_filters", filters)

        self.encoder = nn.Sequential(
            nn.Conv2d(4, 32, 3, padding=1, groups=4, bias=False),
            nn.BatchNorm2d(32),
            nn.SiLU(),
            nn.Conv2d(32, 64, 1, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 64, 3, stride=2, padding=1, groups=64, bias=False),
            nn.BatchNorm2d(64),
            nn.SiLU(),
            nn.Conv2d(64, 128, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.SiLU(),
            nn.AdaptiveAvgPool2d(1)
        )
        self.project = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128, output_dim),
            nn.LayerNorm(output_dim),
            nn.SiLU()
        )

    def forward(self, image):
        gray = image.mean(dim=1, keepdim=True)
        subbands = F.conv2d(gray, self.haar_filters, stride=2)
        return self.project(self.encoder(subbands))

class LiteRCFNet(nn.Module):
    def __init__(self, use_context=True, use_frequency=True, projection_dim=256):
        super().__init__()
        self.use_context = use_context
        self.use_frequency = use_frequency

        try:
            self.backbone = timm.create_model(
                BACKBONE_ID, pretrained=True, num_classes=0, global_pool="avg"
            )
        except Exception:
            self.backbone = timm.create_model(
                "mobilenetv4_conv_small.e2400_r224_in1k",
                pretrained=True, num_classes=0, global_pool="avg"
            )

        feature_dim = self.backbone.num_features
        self.feature_dim = feature_dim

        self.roi_project = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.SiLU(),
            nn.Dropout(0.15)
        )
        self.context_project = nn.Sequential(
            nn.Linear(feature_dim, projection_dim),
            nn.LayerNorm(projection_dim),
            nn.SiLU(),
            nn.Dropout(0.15)
        )
        self.frequency = HaarFrequencyBranch(projection_dim)

        self.gate = nn.Sequential(
            nn.Linear(projection_dim * 3, projection_dim),
            nn.SiLU(),
            nn.Dropout(0.10),
            nn.Linear(projection_dim, 2)
        )

        self.classifier = nn.Sequential(
            nn.Linear(projection_dim, 128),
            nn.LayerNorm(128),
            nn.SiLU(),
            nn.Dropout(0.30),
            nn.Linear(128, 2)
        )
        self.roi_aux = nn.Linear(projection_dim, 2)
        self.context_aux = nn.Linear(projection_dim, 2)

    def forward(self, roi, context):
        paired = torch.cat([roi, context], dim=0)
        paired_features = self.backbone(paired)
        roi_features, context_features = paired_features.chunk(2, dim=0)

        roi_vector = self.roi_project(roi_features)
        context_vector = self.context_project(context_features)

        if not self.use_context:
            context_vector = torch.zeros_like(context_vector)

        if self.use_frequency:
            frequency_vector = self.frequency(roi)
        else:
            frequency_vector = torch.zeros_like(roi_vector)

        gate_values = torch.sigmoid(
            self.gate(torch.cat([roi_vector, context_vector, frequency_vector], dim=1))
        )
        context_gate = gate_values[:, 0:1]
        frequency_gate = gate_values[:, 1:2]

        fused = roi_vector
        if self.use_context:
            fused = fused + context_gate * context_vector
        if self.use_frequency:
            fused = fused + frequency_gate * frequency_vector

        return {
            "logits": self.classifier(fused),
            "roi_logits": self.roi_aux(roi_vector),
            "context_logits": self.context_aux(context_vector),
            "context_gate": context_gate,
            "frequency_gate": frequency_gate
        }

model = LiteRCFNet(
    use_context=EXP["use_context"],
    use_frequency=EXP["use_frequency"]
).to(DEVICE)

total_params = sum(parameter.numel() for parameter in model.parameters())
trainable_params = sum(
    parameter.numel() for parameter in model.parameters() if parameter.requires_grad
)

print("Backbone:", BACKBONE_ID)
print("Backbone feature dimension:", model.feature_dim)
print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")

## Step 12 — Losses, metrics, mixed precision, checkpointing, and training loops

Training loss combines:
- clean classification loss,
- marker-view classification loss,
- ROI/context auxiliary losses,
- Jensen–Shannon consistency between clean and marker-view probabilities.

**Desired output:** training utilities are defined without an error.

In [ ]:
criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS, label_smoothing=0.05)

def js_divergence(logits_a, logits_b):
    prob_a = torch.softmax(logits_a, dim=1)
    prob_b = torch.softmax(logits_b, dim=1)
    mean_prob = 0.5 * (prob_a + prob_b)
    kl_a = F.kl_div(torch.log(prob_a.clamp_min(1e-7)), mean_prob, reduction="batchmean")
    kl_b = F.kl_div(torch.log(prob_b.clamp_min(1e-7)), mean_prob, reduction="batchmean")
    return 0.5 * (kl_a + kl_b)

def compute_training_loss(clean_output, marker_output, labels):
    clean_ce = criterion(clean_output["logits"], labels)
    marker_ce = criterion(marker_output["logits"], labels)

    auxiliary = criterion(clean_output["roi_logits"], labels)
    if EXP["use_context"]:
        auxiliary = auxiliary + criterion(clean_output["context_logits"], labels)

    consistency = js_divergence(
        clean_output["logits"], marker_output["logits"]
    )

    total = (
        clean_ce
        + 0.50 * marker_ce
        + 0.10 * auxiliary
        + EXP["lambda_consistency"] * consistency
    )
    return total, {
        "clean_ce": clean_ce.detach().item(),
        "marker_ce": marker_ce.detach().item(),
        "auxiliary": auxiliary.detach().item(),
        "consistency": consistency.detach().item()
    }

def freeze_batchnorm_statistics(module):
    for layer in module.modules():
        if isinstance(layer, nn.modules.batchnorm._BatchNorm):
            layer.eval()

def metric_bundle(labels, probabilities, threshold=0.5):
    labels = np.asarray(labels).astype(int)
    probabilities = np.asarray(probabilities)
    predictions = (probabilities >= threshold).astype(int)

    matrix = confusion_matrix(labels, predictions, labels=[0, 1])
    tn, fp, fn, tp = matrix.ravel()
    specificity = tn / max(tn + fp, 1)

    return {
        "accuracy": accuracy_score(labels, predictions),
        "precision": precision_score(labels, predictions, zero_division=0),
        "sensitivity": recall_score(labels, predictions, zero_division=0),
        "specificity": specificity,
        "f1": f1_score(labels, predictions, zero_division=0),
        "auroc": roc_auc_score(labels, probabilities),
        "auprc": average_precision_score(labels, probabilities),
        "mcc": matthews_corrcoef(labels, predictions),
        "brier": brier_score_loss(labels, probabilities),
        "confusion_matrix": matrix.tolist()
    }

def autocast_context():
    if DEVICE.type == "cuda":
        return torch.autocast(
            device_type="cuda", dtype=torch.float16, enabled=USE_AMP
        )
    return nullcontext()

try:
    scaler = torch.amp.GradScaler("cuda", enabled=USE_AMP)
except Exception:
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)

def train_one_epoch(model, loader, optimizer):
    model.train()
    freeze_batchnorm_statistics(model.backbone)

    running_loss = 0.0
    all_labels = []
    all_probabilities = []

    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(loader):
        roi = batch["roi"].to(DEVICE, non_blocking=True)
        context = batch["context"].to(DEVICE, non_blocking=True)
        roi_marker = batch["roi_marker"].to(DEVICE, non_blocking=True)
        context_marker = batch["context_marker"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with autocast_context():
            clean_output = model(roi, context)
            marker_output = model(roi_marker, context_marker)
            loss, _ = compute_training_loss(clean_output, marker_output, labels)
            scaled_loss = loss / ACCUM_STEPS

        scaler.scale(scaled_loss).backward()

        should_step = (
            (step + 1) % ACCUM_STEPS == 0
            or (step + 1) == len(loader)
        )
        if should_step:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.detach().item()
        probabilities = torch.softmax(clean_output["logits"], dim=1)[:, 1]
        all_labels.extend(labels.detach().cpu().numpy())
        all_probabilities.extend(probabilities.detach().cpu().numpy())

    metrics = metric_bundle(all_labels, all_probabilities)
    metrics["loss"] = running_loss / max(len(loader), 1)
    return metrics

@torch.inference_mode()
def evaluate_loader(model, loader):
    model.eval()
    running_loss = 0.0
    all_logits = []
    all_labels = []
    all_paths = []
    all_context_gates = []
    all_frequency_gates = []

    for batch in loader:
        roi = batch["roi"].to(DEVICE, non_blocking=True)
        context = batch["context"].to(DEVICE, non_blocking=True)
        labels = batch["label"].to(DEVICE, non_blocking=True)

        with autocast_context():
            output = model(roi, context)
            loss = criterion(output["logits"], labels)

        running_loss += loss.detach().item()
        all_logits.append(output["logits"].float().cpu())
        all_labels.append(labels.cpu())
        all_paths.extend(batch["path"])
        all_context_gates.append(output["context_gate"].float().cpu())
        all_frequency_gates.append(output["frequency_gate"].float().cpu())

    logits = torch.cat(all_logits)
    labels = torch.cat(all_labels)
    probabilities = torch.softmax(logits, dim=1)[:, 1].numpy()
    metrics = metric_bundle(labels.numpy(), probabilities)
    metrics["loss"] = running_loss / max(len(loader), 1)

    return {
        "metrics": metrics,
        "logits": logits,
        "labels": labels,
        "paths": all_paths,
        "context_gates": torch.cat(all_context_gates).numpy().ravel(),
        "frequency_gates": torch.cat(all_frequency_gates).numpy().ravel()
    }

def save_checkpoint(path, model, optimizer, scheduler, epoch, best_auc, history):
    torch.save({
        "epoch": epoch,
        "model_state": model.state_dict(),
        "optimizer_state": optimizer.state_dict(),
        "scheduler_state": scheduler.state_dict() if scheduler is not None else None,
        "best_auc": best_auc,
        "history": history,
        "experiment": EXPERIMENT_NAME,
        "seed": SEED
    }, path)

def load_model_weights(path, model):
    checkpoint = torch.load(path, map_location=DEVICE)
    model.load_state_dict(checkpoint["model_state"])
    return checkpoint

print("Training utilities are ready.")

## Step 13 — Stage controller with automatic resume

Each epoch saves `last` to Google Drive. The best validation-AUROC checkpoint is saved separately.  
After a Colab disconnect, rerun Steps 1–13, then rerun the interrupted training stage.

**Desired output:** the `fit_stage` function is defined.

In [ ]:
def fit_stage(
    model, train_loader, val_loader, optimizer, scheduler,
    epochs, stage_name
):
    last_path = RUN_DIR / f"{stage_name}_last.pt"
    best_path = RUN_DIR / f"{stage_name}_best.pt"

    start_epoch = 0
    best_auc = -np.inf
    patience_counter = 0
    history = []

    if last_path.exists():
        checkpoint = torch.load(last_path, map_location=DEVICE)
        if checkpoint.get("experiment") == EXPERIMENT_NAME and checkpoint.get("seed") == SEED:
            model.load_state_dict(checkpoint["model_state"])
            optimizer.load_state_dict(checkpoint["optimizer_state"])
            if scheduler is not None and checkpoint["scheduler_state"] is not None:
                scheduler.load_state_dict(checkpoint["scheduler_state"])
            start_epoch = checkpoint["epoch"] + 1
            best_auc = checkpoint["best_auc"]
            history = checkpoint.get("history", [])
            print(f"Resuming {stage_name} from epoch {start_epoch + 1}.")

    if start_epoch >= epochs:
        print(f"{stage_name} is already complete.")
        if best_path.exists():
            load_model_weights(best_path, model)
        return history

    for epoch in range(start_epoch, epochs):
        start_time = time.time()

        train_metrics = train_one_epoch(model, train_loader, optimizer)
        val_result = evaluate_loader(model, val_loader)
        val_metrics = val_result["metrics"]

        if scheduler is not None:
            scheduler.step(val_metrics["auroc"])

        epoch_record = {
            "stage": stage_name,
            "epoch": epoch + 1,
            "train_loss": train_metrics["loss"],
            "train_accuracy": train_metrics["accuracy"],
            "train_f1": train_metrics["f1"],
            "train_auroc": train_metrics["auroc"],
            "val_loss": val_metrics["loss"],
            "val_accuracy": val_metrics["accuracy"],
            "val_f1": val_metrics["f1"],
            "val_auroc": val_metrics["auroc"],
            "val_auprc": val_metrics["auprc"],
            "minutes": (time.time() - start_time) / 60
        }
        history.append(epoch_record)

        improved = val_metrics["auroc"] > best_auc + 1e-5
        if improved:
            best_auc = val_metrics["auroc"]
            patience_counter = 0
            save_checkpoint(
                best_path, model, optimizer, scheduler,
                epoch, best_auc, history
            )
        else:
            patience_counter += 1

        save_checkpoint(
            last_path, model, optimizer, scheduler,
            epoch, best_auc, history
        )
        pd.DataFrame(history).to_csv(
            RUN_DIR / f"{stage_name}_history.csv", index=False
        )

        print(
            f"{stage_name} | Epoch {epoch + 1:02d}/{epochs:02d} | "
            f"Train loss {train_metrics['loss']:.4f} | "
            f"Val loss {val_metrics['loss']:.4f} | "
            f"Val Acc {val_metrics['accuracy']:.4f} | "
            f"Val F1 {val_metrics['f1']:.4f} | "
            f"Val AUC {val_metrics['auroc']:.4f} | "
            f"{epoch_record['minutes']:.1f} min"
        )

        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"Early stopping after {EARLY_STOPPING_PATIENCE} non-improving epochs.")
            break

    if best_path.exists():
        load_model_weights(best_path, model)
    return history

print("Stage controller is ready.")

## Step 14 — Stage 1: warm up new branches and classifier

The pretrained MobileNetV4 backbone is frozen. Only ROI/context projection, frequency, fusion, and classification layers learn.

**Desired output:** validation AUROC improves and `stage1_best.pt` is saved in Drive.

In [ ]:
for parameter in model.backbone.parameters():
    parameter.requires_grad = False

head_parameters = [
    parameter for parameter in model.parameters()
    if parameter.requires_grad
]

optimizer_stage1 = torch.optim.AdamW(
    head_parameters, lr=3e-4, weight_decay=1e-4
)
scheduler_stage1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage1, mode="max", factor=0.5, patience=2, min_lr=1e-6
)

history_stage1 = fit_stage(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_stage1,
    scheduler=scheduler_stage1,
    epochs=WARMUP_EPOCHS,
    stage_name="stage1"
)

print("Stage 1 best checkpoint:", RUN_DIR / "stage1_best.pt")

## Step 15 — Stage 2: fine-tune the last 30% of backbone parameters

Backbone learning rate is 10× lower than the new modules. Frozen BatchNorm statistics protect against unstable updates on a small medical dataset.

**Desired output:** the best validation AUROC checkpoint is saved as `stage2_best.pt`.

In [ ]:
if (RUN_DIR / "stage1_best.pt").exists():
    load_model_weights(RUN_DIR / "stage1_best.pt", model)

backbone_parameters = list(model.backbone.parameters())
for parameter in backbone_parameters:
    parameter.requires_grad = False

unfreeze_count = max(1, int(0.30 * len(backbone_parameters)))
for parameter in backbone_parameters[-unfreeze_count:]:
    parameter.requires_grad = True

trainable_backbone = [
    parameter for parameter in model.backbone.parameters()
    if parameter.requires_grad
]
trainable_new_modules = [
    parameter for name, parameter in model.named_parameters()
    if not name.startswith("backbone.") and parameter.requires_grad
]

optimizer_stage2 = torch.optim.AdamW(
    [
        {"params": trainable_backbone, "lr": 1e-5},
        {"params": trainable_new_modules, "lr": 1e-4}
    ],
    weight_decay=1e-4
)
scheduler_stage2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer_stage2, mode="max", factor=0.5, patience=2, min_lr=1e-7
)

print("Trainable backbone tensors:", len(trainable_backbone))
print("Trainable new-module tensors:", len(trainable_new_modules))

history_stage2 = fit_stage(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer_stage2,
    scheduler=scheduler_stage2,
    epochs=FINETUNE_EPOCHS,
    stage_name="stage2"
)

print("Stage 2 best checkpoint:", RUN_DIR / "stage2_best.pt")

## Step 16 — Temperature calibration and validation threshold selection

Temperature is fitted only on validation logits. The decision threshold is also selected only from validation data by maximizing F1. The untouched test set is used afterward.

**Desired output:** learned temperature and validation-selected threshold are printed.

In [ ]:
BEST_MODEL_PATH = (
    RUN_DIR / "stage2_best.pt"
    if (RUN_DIR / "stage2_best.pt").exists()
    else RUN_DIR / "stage1_best.pt"
)
load_model_weights(BEST_MODEL_PATH, model)

val_result = evaluate_loader(model, val_loader)
val_logits = val_result["logits"].to(DEVICE)
val_labels = val_result["labels"].to(DEVICE)

class TemperatureScaler(nn.Module):
    def __init__(self):
        super().__init__()
        self.log_temperature = nn.Parameter(torch.zeros(1, device=DEVICE))

    def forward(self, logits):
        return logits / self.log_temperature.exp().clamp_min(1e-4)

temperature_model = TemperatureScaler().to(DEVICE)
calibration_optimizer = torch.optim.LBFGS(
    temperature_model.parameters(), lr=0.05, max_iter=100
)
calibration_criterion = nn.CrossEntropyLoss()

def calibration_closure():
    calibration_optimizer.zero_grad()
    loss = calibration_criterion(
        temperature_model(val_logits), val_labels
    )
    loss.backward()
    return loss

calibration_optimizer.step(calibration_closure)
TEMPERATURE = float(
    temperature_model.log_temperature.exp().detach().cpu().item()
)

calibrated_val_prob = torch.softmax(
    val_result["logits"] / TEMPERATURE, dim=1
)[:, 1].numpy()
val_targets = val_result["labels"].numpy()

thresholds = np.linspace(0.05, 0.95, 181)
threshold_rows = []
for threshold in thresholds:
    metrics = metric_bundle(val_targets, calibrated_val_prob, threshold)
    threshold_rows.append({
        "threshold": threshold,
        "f1": metrics["f1"],
        "sensitivity": metrics["sensitivity"],
        "specificity": metrics["specificity"]
    })

threshold_df = pd.DataFrame(threshold_rows)
best_row = threshold_df.sort_values(
    ["f1", "specificity"], ascending=False
).iloc[0]
BEST_THRESHOLD = float(best_row["threshold"])

calibration_info = {
    "temperature": TEMPERATURE,
    "validation_threshold": BEST_THRESHOLD,
    "validation_f1": float(best_row["f1"]),
    "validation_sensitivity": float(best_row["sensitivity"]),
    "validation_specificity": float(best_row["specificity"])
}
with open(RUN_DIR / "calibration.json", "w") as handle:
    json.dump(calibration_info, handle, indent=2)

print(json.dumps(calibration_info, indent=2))

## Step 17 — Final untouched TN5000 test evaluation

This reports both default-threshold and validation-tuned-threshold results.

**Desired output:** accuracy, precision, sensitivity, specificity, F1, AUROC, AUPRC, MCC, Brier score, and confusion matrix.

In [ ]:
test_result = evaluate_loader(model, test_loader)
test_logits = test_result["logits"]
test_labels = test_result["labels"].numpy()
test_probabilities = torch.softmax(
    test_logits / TEMPERATURE, dim=1
)[:, 1].numpy()

test_metrics_default = metric_bundle(
    test_labels, test_probabilities, threshold=0.50
)
test_metrics_tuned = metric_bundle(
    test_labels, test_probabilities, threshold=BEST_THRESHOLD
)

print("=" * 72)
print("TN5000 TEST RESULTS — DEFAULT THRESHOLD 0.50")
print("=" * 72)
for key, value in test_metrics_default.items():
    print(f"{key:18s}: {value}")

print("\n" + "=" * 72)
print(f"TN5000 TEST RESULTS — VALIDATION-TUNED THRESHOLD {BEST_THRESHOLD:.3f}")
print("=" * 72)
for key, value in test_metrics_tuned.items():
    print(f"{key:18s}: {value}")

prediction_df = pd.DataFrame({
    "path": test_result["paths"],
    "label": test_labels,
    "malignant_probability": test_probabilities,
    "prediction": (test_probabilities >= BEST_THRESHOLD).astype(int),
    "context_gate": test_result["context_gates"],
    "frequency_gate": test_result["frequency_gates"]
})
prediction_df.to_csv(RUN_DIR / "tn5000_test_predictions.csv", index=False)

final_report = {
    "experiment": EXPERIMENT_NAME,
    "seed": SEED,
    "temperature": TEMPERATURE,
    "threshold": BEST_THRESHOLD,
    "default_threshold_metrics": test_metrics_default,
    "tuned_threshold_metrics": test_metrics_tuned
}
with open(RUN_DIR / "tn5000_test_metrics.json", "w") as handle:
    json.dump(final_report, handle, indent=2)

## Step 18 — Confusion matrix, ROC, PR, and reliability plots

All figures are saved to Google Drive.

**Desired output:** four separate plots.

In [ ]:
def expected_calibration_error(labels, probabilities, bins=10):
    labels = np.asarray(labels)
    probabilities = np.asarray(probabilities)
    edges = np.linspace(0, 1, bins + 1)
    ece = 0.0
    table = []

    for left, right in zip(edges[:-1], edges[1:]):
        mask = (probabilities > left) & (probabilities <= right)
        if not mask.any():
            continue
        confidence = probabilities[mask].mean()
        accuracy = labels[mask].mean()
        weight = mask.mean()
        ece += weight * abs(accuracy - confidence)
        table.append((left, right, confidence, accuracy, mask.sum()))
    return float(ece), table

matrix = np.array(test_metrics_tuned["confusion_matrix"])
plt.figure(figsize=(5, 4))
plt.imshow(matrix)
plt.xticks([0, 1], ["Benign", "Malignant"])
plt.yticks([0, 1], ["Benign", "Malignant"])
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("TN5000 Test Confusion Matrix")
for i in range(2):
    for j in range(2):
        plt.text(j, i, matrix[i, j], ha="center", va="center")
plt.tight_layout()
plt.savefig(RUN_DIR / "confusion_matrix.png", dpi=220)
plt.show()

fpr, tpr, _ = roc_curve(test_labels, test_probabilities)
plt.figure(figsize=(5, 4))
plt.plot(fpr, tpr, label=f"AUC = {test_metrics_tuned['auroc']:.4f}")
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("TN5000 Test ROC")
plt.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "roc_curve.png", dpi=220)
plt.show()

precision_values, recall_values, _ = precision_recall_curve(
    test_labels, test_probabilities
)
plt.figure(figsize=(5, 4))
plt.plot(
    recall_values, precision_values,
    label=f"AUPRC = {test_metrics_tuned['auprc']:.4f}"
)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("TN5000 Test Precision–Recall Curve")
plt.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "pr_curve.png", dpi=220)
plt.show()

ece, reliability_rows = expected_calibration_error(
    test_labels, test_probabilities, bins=10
)
reliability_df = pd.DataFrame(
    reliability_rows,
    columns=["bin_left", "bin_right", "confidence", "observed_rate", "count"]
)
plt.figure(figsize=(5, 4))
plt.plot(
    reliability_df["confidence"],
    reliability_df["observed_rate"],
    marker="o"
)
plt.plot([0, 1], [0, 1], linestyle="--")
plt.xlabel("Mean predicted probability")
plt.ylabel("Observed malignant rate")
plt.title(f"Reliability Diagram — ECE {ece:.4f}")
plt.tight_layout()
plt.savefig(RUN_DIR / "reliability_diagram.png", dpi=220)
plt.show()

print("Expected calibration error:", ece)

## Step 19 — Training curves

**Desired output:** loss and validation-AUROC plots saved to Drive.

In [ ]:
history_frames = []
for history_file in [
    RUN_DIR / "stage1_history.csv",
    RUN_DIR / "stage2_history.csv"
]:
    if history_file.exists():
        history_frames.append(pd.read_csv(history_file))

history_df = pd.concat(history_frames, ignore_index=True)
history_df["global_epoch"] = np.arange(1, len(history_df) + 1)
history_df.to_csv(RUN_DIR / "complete_training_history.csv", index=False)

plt.figure(figsize=(7, 4))
plt.plot(history_df["global_epoch"], history_df["train_loss"], label="Train loss")
plt.plot(history_df["global_epoch"], history_df["val_loss"], label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training and Validation Loss")
plt.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "loss_curve.png", dpi=220)
plt.show()

plt.figure(figsize=(7, 4))
plt.plot(history_df["global_epoch"], history_df["val_auroc"], label="Validation AUROC")
plt.plot(history_df["global_epoch"], history_df["val_f1"], label="Validation F1")
plt.xlabel("Epoch")
plt.ylabel("Score")
plt.title("Validation Performance")
plt.legend()
plt.tight_layout()
plt.savefig(RUN_DIR / "validation_performance.png", dpi=220)
plt.show()

## Step 20 — Parameter count, model size, approximate FLOPs, and T4 latency

Latency is measured with batch size 1 after warm-up. FLOPs are approximate and depend on profiler coverage.

**Desired output:** parameters, model size, GFLOPs, and milliseconds per image.

In [ ]:
class InferenceWrapper(nn.Module):
    def __init__(self, base_model):
        super().__init__()
        self.base_model = base_model

    def forward(self, roi, context):
        return self.base_model(roi, context)["logits"]

model.eval()
wrapper = InferenceWrapper(model).to(DEVICE).eval()

dummy_roi = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)
dummy_context = torch.randn(1, 3, IMAGE_SIZE, IMAGE_SIZE, device=DEVICE)

try:
    macs, profiled_parameters = profile(
        wrapper, inputs=(dummy_roi, dummy_context), verbose=False
    )
    gflops = 2 * macs / 1e9
except Exception as error:
    print("FLOP profiling warning:", error)
    gflops = float("nan")

temporary_model_path = RUN_DIR / "inference_model_state.pt"
torch.save(model.state_dict(), temporary_model_path)
model_size_mb = temporary_model_path.stat().st_size / 2**20

with torch.inference_mode():
    for _ in range(20):
        _ = wrapper(dummy_roi, dummy_context)
    torch.cuda.synchronize()

    times = []
    for _ in range(100):
        start = time.perf_counter()
        _ = wrapper(dummy_roi, dummy_context)
        torch.cuda.synchronize()
        times.append((time.perf_counter() - start) * 1000)

efficiency = {
    "total_parameters": int(sum(p.numel() for p in model.parameters())),
    "trainable_parameters_final": int(
        sum(p.numel() for p in model.parameters() if p.requires_grad)
    ),
    "model_size_mb": float(model_size_mb),
    "approximate_gflops": float(gflops),
    "t4_latency_ms_median_batch1": float(np.median(times)),
    "t4_latency_ms_mean_batch1": float(np.mean(times))
}
with open(RUN_DIR / "efficiency.json", "w") as handle:
    json.dump(efficiency, handle, indent=2)

print(json.dumps(efficiency, indent=2))

## Step 21 — External dataset evaluation

This is a real dataset-shift test. No external image is used for training, threshold selection, or temperature calibration.

Because the external dataset has no bounding boxes, the full image is used as both ROI and context. State this limitation clearly in the paper.

**Desired output:** external accuracy, F1, sensitivity, specificity, AUROC, and AUPRC.

In [ ]:
external_result = evaluate_loader(model, external_loader)
external_labels = external_result["labels"].numpy()
external_probabilities = torch.softmax(
    external_result["logits"] / TEMPERATURE, dim=1
)[:, 1].numpy()

external_metrics = metric_bundle(
    external_labels, external_probabilities, threshold=BEST_THRESHOLD
)

print("=" * 72)
print("EXTERNAL DATASET RESULTS")
print("=" * 72)
for key, value in external_metrics.items():
    print(f"{key:18s}: {value}")

external_prediction_df = pd.DataFrame({
    "path": external_result["paths"],
    "label": external_labels,
    "malignant_probability": external_probabilities,
    "prediction": (external_probabilities >= BEST_THRESHOLD).astype(int)
})
external_prediction_df.to_csv(
    RUN_DIR / "external_predictions.csv", index=False
)

with open(RUN_DIR / "external_metrics.json", "w") as handle:
    json.dump(external_metrics, handle, indent=2)

## Step 22 — Lightweight input-gradient explanation

This produces a model-agnostic saliency image for the lesion ROI. It is not a replacement for clinical validation, but it helps identify whether predictions rely on the nodule or unrelated image regions.

**Desired output:** original ROI and saliency overlay for one test image.

In [ ]:
model.eval()
explanation_sample = test_dataset[random.randrange(len(test_dataset))]
roi = explanation_sample["roi"].unsqueeze(0).to(DEVICE)
context = explanation_sample["context"].unsqueeze(0).to(DEVICE)
roi.requires_grad_(True)

output = model(roi, context)
predicted_class = output["logits"].argmax(dim=1).item()
score = output["logits"][0, predicted_class]

model.zero_grad(set_to_none=True)
score.backward()

saliency = roi.grad.detach().abs().mean(dim=1)[0].cpu().numpy()
saliency = (saliency - saliency.min()) / (saliency.max() - saliency.min() + 1e-8)
roi_image = denormalize(explanation_sample["roi"])

plt.figure(figsize=(5, 4))
plt.imshow(roi_image)
plt.imshow(saliency, alpha=0.45)
plt.title(
    f"Input-gradient saliency | Predicted: "
    f"{'Malignant' if predicted_class == 1 else 'Benign'}"
)
plt.axis("off")
plt.tight_layout()
plt.savefig(RUN_DIR / "sample_saliency.png", dpi=220)
plt.show()

## Step 23 — Final saved-output audit

**Desired output:** the Drive folder contains checkpoints, CSV predictions, JSON metrics, and plots.

In [ ]:
print("Saved output folder:", RUN_DIR)
for path in sorted(RUN_DIR.iterdir()):
    if path.is_file():
        print(f"{path.name:40s} {path.stat().st_size / 1024:.1f} KB")

# Required research runs

Run the notebook separately for these experiments by changing `EXPERIMENT_NAME`:

| Run | Configuration | Purpose |
|---|---|---|
| A0 | `A0_roi_only` | Lightweight backbone + lesion ROI |
| A1 | `A1_roi_context` | Contribution of peri-lesional context |
| A2 | `A2_roi_frequency` | Contribution of Haar frequency features |
| A3 | `A3_no_consistency` | Context + frequency without marker invariance |
| Full | `full` | Complete LiteRCF-Net |

For the final paper:
1. run all five ablations with seed 42;
2. run the full model with seeds 42, 43, and 44;
3. report mean ± standard deviation for the full model;
4. keep the official TN5000 test set untouched;
5. report the external evaluation separately;
6. never claim guaranteed accuracy before the experiments are completed.